In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader

docs = CSVLoader(
    file_path='../../data/raw/hyperbook.csv', 
    content_columns=['content'], 
    metadata_columns=['url', 'title', 'authors'],
    encoding='utf-8'
).load()

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk parameters
chunk_size = 256
chunk_overlap = 32

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
doc_splits = text_splitter.split_documents(docs)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")
vectorstore = InMemoryVectorStore.from_documents(documents=doc_splits, embedding=embeddings)
vectorstore.dump(f'../../embeddings/hyperbook_embeddings_{chunk_size}_{chunk_overlap}')

d:\Projects\summer-job\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

# Chunk parameters
chunk_size, chunk_overlap = 256, 32
embedding_model = OllamaEmbeddings(model='dengcao/Qwen3-Embedding-0.6B:Q8_0')
embeddings_path = f'../../embeddings/hyperbook_embeddings_{chunk_size}_{chunk_overlap}'

# Loading the vector store locally
vectorstore = InMemoryVectorStore.load(path=embeddings_path, embedding=embedding_model)
dense_retriever = vectorstore.as_retriever()